In [51]:
import sys
import subprocess
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBRegressor

# Load dataset and keep an untouched copy for repeatable mappings
df_raw = pd.read_csv("crop_dataset.csv")
df = df_raw.copy()

In [86]:
df.columns

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_x0020_Price', 'Max_x0020_Price',
       'Modal_x0020_Price', 'City', 'Crop Type', 'Season',
       'Temperature (degree C)', 'Rainfall (mm)', 'Supply Volume (tons)',
       'Demand Volume (tons)', 'Transportation Cost (₹/ton)',
       'Fertilizer Usage (kg/hectare)', 'Pest Infestation (0-1)',
       'Market Competition (0-1)', 'Price (₹/ton)'],
      dtype='object')

In [48]:
# Drop the 'Unnamed: 0' and 'Date' columns
df = df.drop(columns=['Unnamed: 0', 'Date'], errors='ignore')

In [87]:
df.isnull().sum()

State                            0
District                         0
Market                           0
Commodity                        0
Variety                          0
Grade                            0
Arrival_Date                     0
Min_x0020_Price                  0
Max_x0020_Price                  0
Modal_x0020_Price                0
City                             0
Crop Type                        0
Season                           0
Temperature (degree C)           0
Rainfall (mm)                    0
Supply Volume (tons)             0
Demand Volume (tons)             0
Transportation Cost (₹/ton)      0
Fertilizer Usage (kg/hectare)    0
Pest Infestation (0-1)           0
Market Competition (0-1)         0
Price (₹/ton)                    0
dtype: int64

In [88]:
# Identify object-type columns in the current dataframe
object_cols = df.select_dtypes(include=["object"]).columns

# Build mappings only when object columns are present
if len(object_cols) > 0:
    mappings = {}

    for col in object_cols:
        # Sort values for stable mapping and skip missing values
        unique_values = sorted(df[col].dropna().unique())
        mapping = {value: idx for idx, value in enumerate(unique_values)}

        # Store mapping and apply encoding
        mappings[col] = mapping
        df[col] = df[col].map(mapping)

# Print mappings if available (works even after re-running cells)
if "mappings" in globals() and len(mappings) > 0:
    for col, mapping in mappings.items():
        print(f"Mapping for {col}:")
        for value, idx in mapping.items():
            print(f"  {value}: {idx}")
        print()
else:
    print("No object-type columns found and no previous mappings available.")

Mapping for State:
  Andhra Pradesh: 0
  Assam: 1
  Bihar: 2
  Chandigarh: 3
  Gujarat: 4
  Haryana: 5
  Himachal Pradesh: 6
  Kerala: 7
  Madhya Pradesh: 8
  Nagaland: 9
  Odisha: 10
  Punjab: 11
  Rajasthan: 12
  Tamil Nadu: 13
  Telangana: 14
  Tripura: 15
  Uttar Pradesh: 16
  Uttarakhand: 17
  West Bengal: 18

Mapping for District:
  Adilabad: 0
  Ahmedabad: 1
  Ajmer: 2
  Alappuzha: 3
  Aligarh: 4
  Alwar: 5
  Amreli: 6
  Amritsar: 7
  Annamayya: 8
  Ariyalur: 9
  Badaun: 10
  Badwani: 11
  Bahraich: 12
  Balrampur: 13
  Bargarh: 14
  Beawar: 15
  Bhadradri Kothagudem: 16
  Bhatinda: 17
  Bhiwani: 18
  Bikaner: 19
  Birbhum: 20
  Boudh: 21
  Chamba: 22
  Chandigarh: 23
  Chengalpattu: 24
  Chhota Udaipur: 25
  Chittor: 26
  Chittorgarh: 27
  Churu: 28
  Coimbatore: 29
  Coochbehar: 30
  Cuddalore: 31
  Cuttack: 32
  Darjeeling: 33
  Deeg: 34
  Dehradoon: 35
  Dhar: 36
  Dharmapuri: 37
  Dhenkanal: 38
  East Champaran/ Motihari: 39
  East Godavari: 40
  Ernakulam: 41
  Erode: 42
 

In [89]:
# Display mappings for requested categories with fallback column names
if "mappings" not in globals() or len(mappings) == 0:
    print("Mappings not found. Run the encoding cell first.")
else:
    # Requested labels mapped to possible dataset column names
    requested = {
        "State": ["State"],
        "City": ["City", "District", "Market"],
        "Crop": ["Crop", "Crops", "Commodity"],
        "Season": ["Season", "Arrival_Season"],
    }

    shown = []

    for label, candidates in requested.items():
        matched = [c for c in candidates if c in mappings]

        if len(matched) == 0:
            print(f"{label}: not present in dataset")
            print()
            continue

        for col in matched:
            shown.append(col)
            print(f"Mapping for {label} (column: {col}):")
            for value, idx in mappings[col].items():
                print(f"  {value}: {idx}")
            print()

    # Print remaining mapping columns as other details
    remaining_cols = [col for col in mappings.keys() if col not in shown]
    if len(remaining_cols) > 0:
        print("Other mapping details:")
        print()
        for col in remaining_cols:
            print(f"Mapping for {col}:")
            for value, idx in mappings[col].items():
                print(f"  {value}: {idx}")
            print()

Mapping for State (column: State):
  Andhra Pradesh: 0
  Assam: 1
  Bihar: 2
  Chandigarh: 3
  Gujarat: 4
  Haryana: 5
  Himachal Pradesh: 6
  Kerala: 7
  Madhya Pradesh: 8
  Nagaland: 9
  Odisha: 10
  Punjab: 11
  Rajasthan: 12
  Tamil Nadu: 13
  Telangana: 14
  Tripura: 15
  Uttar Pradesh: 16
  Uttarakhand: 17
  West Bengal: 18

Mapping for City (column: City):
  Adilabad: 0
  Ahmedabad: 1
  Ajmer: 2
  Alappuzha: 3
  Aligarh: 4
  Alwar: 5
  Amreli: 6
  Amritsar: 7
  Annamayya: 8
  Ariyalur: 9
  Badaun: 10
  Badwani: 11
  Bahraich: 12
  Balrampur: 13
  Bargarh: 14
  Beawar: 15
  Bhadradri Kothagudem: 16
  Bhatinda: 17
  Bhiwani: 18
  Bikaner: 19
  Birbhum: 20
  Boudh: 21
  Chamba: 22
  Chandigarh: 23
  Chengalpattu: 24
  Chhota Udaipur: 25
  Chittor: 26
  Chittorgarh: 27
  Churu: 28
  Coimbatore: 29
  Coochbehar: 30
  Cuddalore: 31
  Cuttack: 32
  Darjeeling: 33
  Deeg: 34
  Dehradoon: 35
  Dhar: 36
  Dharmapuri: 37
  Dhenkanal: 38
  East Champaran/ Motihari: 39
  East Godavari: 40
  

In [90]:
object_cols

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'City', 'Crop Type', 'Season'],
      dtype='object')

In [91]:
print('df columns:', list(df.columns))
print('mapping keys:', list(mappings.keys()) if 'mappings' in globals() else [])

for token in ['state', 'city', 'district', 'market', 'crop', 'commodity', 'season']:
    cols = [c for c in (list(mappings.keys()) if 'mappings' in globals() else []) if token in c.lower()]
    print(f"{token} ->", cols)

df columns: ['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade', 'Arrival_Date', 'Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price', 'City', 'Crop Type', 'Season', 'Temperature (degree C)', 'Rainfall (mm)', 'Supply Volume (tons)', 'Demand Volume (tons)', 'Transportation Cost (₹/ton)', 'Fertilizer Usage (kg/hectare)', 'Pest Infestation (0-1)', 'Market Competition (0-1)', 'Price (₹/ton)']
mapping keys: ['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade', 'Arrival_Date', 'City', 'Crop Type', 'Season']
state -> ['State']
city -> ['City']
district -> ['District']
market -> ['Market']
crop -> ['Crop Type']
commodity -> ['Commodity']
season -> ['Season']


In [92]:
df.head()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price,City,Crop Type,Season,Temperature (degree C),Rainfall (mm),Supply Volume (tons),Demand Volume (tons),Transportation Cost (₹/ton),Fertilizer Usage (kg/hectare),Pest Infestation (0-1),Market Competition (0-1),Price (₹/ton)
0,12,71,132,34,149,9,0,6000,10000,8000,71,34,0,0.602041,0.179395,0.408109,0.404490,0.621784,0.508276,0.041,0.127,8000
1,12,71,132,45,149,9,0,800,1200,1000,71,45,0,0.428571,0.464697,0.223658,0.178634,0.097770,0.304138,0.465,0.544,1000
2,12,71,132,81,149,9,0,3000,7000,6000,71,81,0,0.658163,0.157781,0.317888,0.286433,0.433105,0.531034,0.763,0.181,6000
3,12,71,132,90,149,8,0,6000,7500,7000,71,90,0,0.683673,0.000000,0.494988,0.364235,0.526587,0.700000,0.669,0.419,7000
4,12,15,42,14,149,8,0,4800,5909,5300,15,14,0,0.311224,0.129683,0.310760,0.289917,0.429674,0.480000,0.538,0.706,5300


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7611 entries, 0 to 7610
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   State                          7611 non-null   int64  
 1   District                       7611 non-null   int64  
 2   Market                         7611 non-null   int64  
 3   Commodity                      7611 non-null   int64  
 4   Variety                        7611 non-null   int64  
 5   Grade                          7611 non-null   int64  
 6   Arrival_Date                   7611 non-null   int64  
 7   Min_x0020_Price                7611 non-null   int64  
 8   Max_x0020_Price                7611 non-null   int64  
 9   Modal_x0020_Price              7611 non-null   int64  
 10  City                           7611 non-null   int64  
 11  Crop Type                      7611 non-null   int64  
 12  Season                         7611 non-null   i

In [94]:
# model tarined before normailsation

In [52]:
# Prepare features and target
target_col = "Price (₹/ton)" if "Price (₹/ton)" in df.columns else "Modal_x0020_Price"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split data into training, validation, and testing sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Target used: {target_col}")
print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Target used: Price (₹/ton)
Train shape: (4566, 21), Val shape: (1522, 21), Test shape: (1523, 21)


In [95]:
df.columns

Index(['State', 'District', 'Market', 'Commodity', 'Variety', 'Grade',
       'Arrival_Date', 'Min_x0020_Price', 'Max_x0020_Price',
       'Modal_x0020_Price', 'City', 'Crop Type', 'Season',
       'Temperature (degree C)', 'Rainfall (mm)', 'Supply Volume (tons)',
       'Demand Volume (tons)', 'Transportation Cost (₹/ton)',
       'Fertilizer Usage (kg/hectare)', 'Pest Infestation (0-1)',
       'Market Competition (0-1)', 'Price (₹/ton)'],
      dtype='object')

In [96]:
# Drop the 'Unnamed: 0' and 'Date' columns
df = df.drop(columns=['Unnamed: 0', 'Date'], errors='ignore')

In [97]:
df.isnull().sum()

State                            0
District                         0
Market                           0
Commodity                        0
Variety                          0
Grade                            0
Arrival_Date                     0
Min_x0020_Price                  0
Max_x0020_Price                  0
Modal_x0020_Price                0
City                             0
Crop Type                        0
Season                           0
Temperature (degree C)           0
Rainfall (mm)                    0
Supply Volume (tons)             0
Demand Volume (tons)             0
Transportation Cost (₹/ton)      0
Fertilizer Usage (kg/hectare)    0
Pest Infestation (0-1)           0
Market Competition (0-1)         0
Price (₹/ton)                    0
dtype: int64

In [56]:
# Initialize and train the XGBoost model
# Encode categorical columns so XGBoost receives numeric inputs only.
X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_val_enc = pd.get_dummies(X_val, drop_first=False).reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc = pd.get_dummies(X_test, drop_first=False).reindex(columns=X_train_enc.columns, fill_value=0)

model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
)
model.fit(X_train_enc, y_train)

# Make predictions
y_pred_train = model.predict(X_train_enc)
y_pred_val = model.predict(X_val_enc)
y_pred_test = model.predict(X_test_enc)

# Evaluate the model (compatible across sklearn versions)
train_mse = mean_squared_error(y_train, y_pred_train)
val_mse = mean_squared_error(y_val, y_pred_val)
test_mse = mean_squared_error(y_test, y_pred_test)

train_rmse = train_mse ** 0.5
val_rmse = val_mse ** 0.5
test_rmse = test_mse ** 0.5

train_r2 = r2_score(y_train, y_pred_train)
val_r2 = r2_score(y_val, y_pred_val)
test_r2 = r2_score(y_test, y_pred_test)

print(f"Train RMSE: {train_rmse:.2f}, R2: {train_r2:.4f}")
print(f"Val   RMSE: {val_rmse:.2f}, R2: {val_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.2f}, R2: {test_r2:.4f}")

Train RMSE: 22.70, R2: 1.0000
Val   RMSE: 850.72, R2: 0.9691
Test  RMSE: 1393.58, R2: 0.9710


In [57]:
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

print(f"Training R^2: {train_r2:.2f}")
print(f"Validation R^2: {val_r2:.2f}")
print(f"Test R^2: {test_r2:.2f}")

Train RMSE: 22.70
Validation RMSE: 850.72
Test RMSE: 1393.58


In [98]:
# data normalisation models are built

In [62]:
# Assume df is your DataFrame
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Create a list of input columns to be normalized
input_columns = ['Temperature (degree C)', 'Rainfall (mm)', 'Supply Volume (tons)',
                 'Demand Volume (tons)', 'Transportation Cost (₹/ton)',
                 'Fertilizer Usage (kg/hectare)', 'Pest Infestation (0-1)',
                 'Market Competition (0-1)']

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Apply the scaler to the input columns
df[input_columns] = scaler.fit_transform(df[input_columns])

# Now df contains the normalized input columns
print(df.head())

       State District  ... Market Competition (0-1) Price (₹/ton)
0  Rajasthan  Jodhpur  ...                    0.127          8000
1  Rajasthan  Jodhpur  ...                    0.544          1000
2  Rajasthan  Jodhpur  ...                    0.181          6000
3  Rajasthan  Jodhpur  ...                    0.419          7000
4  Rajasthan   Beawar  ...                    0.706          5300

[5 rows x 22 columns]


In [99]:
df.tail()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_x0020_Price,Max_x0020_Price,Modal_x0020_Price,City,Crop Type,Season,Temperature (degree C),Rainfall (mm),Supply Volume (tons),Demand Volume (tons),Transportation Cost (₹/ton),Fertilizer Usage (kg/hectare),Pest Infestation (0-1),Market Competition (0-1),Price (₹/ton)
7606,6,77,290,18,34,0,0,6800,7200,7000,77,18,0,0.540816,0.185159,0.169080,0.076640,0.505146,0.146897,0.918,0.361,7000
7607,6,77,290,55,79,0,0,11000,13000,12000,77,55,0,0.413265,0.154899,0.205391,0.209406,0.188679,0.683793,0.202,0.195,12000
7608,6,77,290,39,53,0,0,5200,5800,5500,77,39,0,0.795918,0.572046,0.388728,0.417650,0.372213,0.311379,0.547,0.562,5500
7609,6,77,290,81,114,0,0,13000,14000,13500,77,81,0,0.561224,0.000000,0.217198,0.169731,0.275300,0.462759,0.625,0.902,13500
7610,7,166,169,44,58,1,0,7000,7000,7000,166,44,0,0.551020,0.317003,0.391847,0.268821,0.342196,0.397241,0.034,0.463,7000


In [100]:
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

print(f"Training R^2: {train_r2:.2f}")
print(f"Validation R^2: {val_r2:.2f}")
print(f"Test R^2: {test_r2:.2f}")

Training RMSE: 1373.50
Validation RMSE: 5446.75
Test RMSE: 8551.50
Training R^2: 0.94
Validation R^2: -0.27
Test R^2: -0.09


In [101]:
# wrapper technique for feature selection

In [71]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import MinMaxScaler

# Assuming df is your DataFrame

# Define input and output columns
X = df.drop(columns=['Price (₹/ton)'])
y = df['Price (₹/ton)']

# Convert categorical columns to numeric first
X_encoded = pd.get_dummies(X, drop_first=False)

# Normalize the input features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_encoded)

# Use a subset for faster RFE execution
max_rows = min(2500, X_scaled.shape[0])
X_rfe = X_scaled[:max_rows]
y_rfe = y.iloc[:max_rows]

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_rfe, y_rfe, test_size=0.2, random_state=42)

# Initialize the model
model = LinearRegression()

# Initialize RFE with faster elimination steps
n_features = min(5, X_encoded.shape[1])
rfe = RFE(estimator=model, n_features_to_select=n_features, step=0.2)

# Fit RFE
rfe.fit(X_train, y_train)

# Get the selected features
selected_features = X_encoded.columns[rfe.support_]

print("Selected Features:")
print(selected_features)

Selected Features:
Index(['Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price',
       'State_Andhra Pradesh', 'State_Chandigarh'],
      dtype='object')


In [72]:
# Manually select the desired input features
selected_features = ['Rainfall (mm)', 'Supply Volume (tons)', 'Demand Volume (tons)',
                     'Transportation Cost (₹/ton)', 'Market Competition (0-1)']

# Prepare features and target using only the selected features
X = df[selected_features]
y = df['Price (₹/ton)']

In [73]:
# Prepare features and target
# X = df.drop(columns=['Price (₹/ton)'])
# y = df['Price (₹/ton)']

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [74]:
# Initialize and train the XGBoost model
model = XGBRegressor(objective='reg:squarederror', eval_metric='rmse')
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [75]:
# Initialize and train the XGBoost model
model = XGBRegressor(objective='reg:squarederror', eval_metric='rmse', random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Evaluate the model
train_mse = mean_squared_error(y_train, y_pred_train)
val_mse = mean_squared_error(y_val, y_pred_val)
test_mse = mean_squared_error(y_test, y_pred_test)

train_rmse = train_mse ** 0.5
val_rmse = val_mse ** 0.5
test_rmse = test_mse ** 0.5

train_r2 = r2_score(y_train, y_pred_train)
val_r2 = r2_score(y_val, y_pred_val)
test_r2 = r2_score(y_test, y_pred_test)

In [76]:
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

print(f"Training R^2: {train_r2:.2f}")
print(f"Validation R^2: {val_r2:.2f}")
print(f"Test R^2: {test_r2:.2f}")

Training RMSE: 1608.84
Validation RMSE: 5394.40
Test RMSE: 8465.71
Training R^2: 0.92
Validation R^2: -0.24
Test R^2: -0.07


In [102]:
# lasso

In [77]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Assuming df is your DataFrame

# Prepare features and target
X = df.drop(columns=['Price (₹/ton)'])
y = df['Price (₹/ton)']

# Convert categorical columns to numeric first
X_encoded = pd.get_dummies(X, drop_first=False)

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Perform Lasso regression for feature selection
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# Get the selected features (non-zero coefficients)
selected_features = X_train.columns[lasso.coef_ != 0]

print("Selected features using Lasso:")
print(selected_features)

Selected features using Lasso:
Index(['Min_x0020_Price', 'Max_x0020_Price', 'Modal_x0020_Price',
       'Temperature (degree C)', 'Rainfall (mm)', 'Demand Volume (tons)',
       'Transportation Cost (₹/ton)', 'Fertilizer Usage (kg/hectare)',
       'Pest Infestation (0-1)', 'Market Competition (0-1)',
       ...
       'Crop Type_Marigold(loose)', 'Crop Type_Onion',
       'Crop Type_Pea Pod/Pea Cod/हरी मटर', 'Crop Type_Peas(Dry)',
       'Crop Type_Pineapple', 'Crop Type_Strawberry',
       'Crop Type_Sweet Pumpkin', 'Crop Type_Taramira', 'Crop Type_Tinda',
       'Crop Type_Tube Rose(Loose)'],
      dtype='object', length=325)


In [78]:
# Manually select the desired input features
selected_features = ['Temperature (degree C)', 'Rainfall (mm)', 'Supply Volume (tons)',
                     'Demand Volume (tons)', 'Transportation Cost (₹/ton)',
                     'Fertilizer Usage (kg/hectare)', 'Pest Infestation (0-1)',
                     'Market Competition (0-1)']

# Prepare features and target using only the selected features
X = df[selected_features]
y = df['Price (₹/ton)']

In [79]:
# Prepare features and target
# X = df.drop(columns=['Price (₹/ton)'])
# y = df['Price (₹/ton)']

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [80]:
# Initialize and train the XGBoost model
model = XGBRegressor(objective='reg:squarederror', eval_metric='rmse', random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Evaluate the model
train_mse = mean_squared_error(y_train, y_pred_train)
val_mse = mean_squared_error(y_val, y_pred_val)
test_mse = mean_squared_error(y_test, y_pred_test)

train_rmse = train_mse ** 0.5
val_rmse = val_mse ** 0.5
test_rmse = test_mse ** 0.5

train_r2 = r2_score(y_train, y_pred_train)
val_r2 = r2_score(y_val, y_pred_val)
test_r2 = r2_score(y_test, y_pred_test)

In [81]:
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

print(f"Training R^2: {train_r2:.2f}")
print(f"Validation R^2: {val_r2:.2f}")
print(f"Test R^2: {test_r2:.2f}")

Training RMSE: 1373.50
Validation RMSE: 5446.75
Test RMSE: 8551.50
Training R^2: 0.94
Validation R^2: -0.27
Test R^2: -0.09


In [103]:
import pickle

# Save the trained model to a file using pickle
with open('xgboost_model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [104]:
# Load the model from the file
with open('xgboost_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# Now you can use 'loaded_model' to make predictions

In [82]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42, n_estimators=200)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_train_rf = rf_model.predict(X_train)
y_pred_val_rf = rf_model.predict(X_val)
y_pred_test_rf = rf_model.predict(X_test)

# Evaluate the model
train_mse_rf = mean_squared_error(y_train, y_pred_train_rf)
val_mse_rf = mean_squared_error(y_val, y_pred_val_rf)
test_mse_rf = mean_squared_error(y_test, y_pred_test_rf)

train_rmse_rf = train_mse_rf ** 0.5
val_rmse_rf = val_mse_rf ** 0.5
test_rmse_rf = test_mse_rf ** 0.5

train_r2_rf = r2_score(y_train, y_pred_train_rf)
val_r2_rf = r2_score(y_val, y_pred_val_rf)
test_r2_rf = r2_score(y_test, y_pred_test_rf)

print(f"Random Forest - Training RMSE: {train_rmse_rf:.2f}")
print(f"Random Forest - Validation RMSE: {val_rmse_rf:.2f}")
print(f"Random Forest - Test RMSE: {test_rmse_rf:.2f}")

print(f"Random Forest - Training R^2: {train_r2_rf:.2f}")
print(f"Random Forest - Validation R^2: {val_r2_rf:.2f}")
print(f"Random Forest - Test R^2: {test_r2_rf:.2f}")

Random Forest - Training RMSE: 2239.00
Random Forest - Validation RMSE: 5134.64
Random Forest - Test RMSE: 8283.41
Random Forest - Training R^2: 0.85
Random Forest - Validation R^2: -0.13
Random Forest - Test R^2: -0.02


In [83]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Linear Regressor
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_train_lr = lr_model.predict(X_train)
y_pred_val_lr = lr_model.predict(X_val)
y_pred_test_lr = lr_model.predict(X_test)

# Evaluate the model
train_mse_lr = mean_squared_error(y_train, y_pred_train_lr)
val_mse_lr = mean_squared_error(y_val, y_pred_val_lr)
test_mse_lr = mean_squared_error(y_test, y_pred_test_lr)

train_rmse_lr = train_mse_lr ** 0.5
val_rmse_lr = val_mse_lr ** 0.5
test_rmse_lr = test_mse_lr ** 0.5

train_r2_lr = r2_score(y_train, y_pred_train_lr)
val_r2_lr = r2_score(y_val, y_pred_val_lr)
test_r2_lr = r2_score(y_test, y_pred_test_lr)

print(f"Linear Regression - Training RMSE: {train_rmse_lr:.2f}")
print(f"Linear Regression - Validation RMSE: {val_rmse_lr:.2f}")
print(f"Linear Regression - Test RMSE: {test_rmse_lr:.2f}")

print(f"Linear Regression - Training R^2: {train_r2_lr:.2f}")
print(f"Linear Regression - Validation R^2: {val_r2_lr:.2f}")
print(f"Linear Regression - Test R^2: {test_r2_lr:.2f}")

Linear Regression - Training RMSE: 5819.87
Linear Regression - Validation RMSE: 4838.31
Linear Regression - Test RMSE: 8194.14
Linear Regression - Training R^2: 0.00
Linear Regression - Validation R^2: 0.00
Linear Regression - Test R^2: -0.00


In [84]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the SVR
svr_model = SVR(C=5.0, epsilon=0.1, kernel='rbf')

# Fit on a subset for faster runtime on large data
subset_size = min(2500, len(X_train))
svr_model.fit(X_train[:subset_size], y_train.iloc[:subset_size])

# Make predictions
y_pred_train_svr = svr_model.predict(X_train)
y_pred_val_svr = svr_model.predict(X_val)
y_pred_test_svr = svr_model.predict(X_test)

# Evaluate the model
train_mse_svr = mean_squared_error(y_train, y_pred_train_svr)
val_mse_svr = mean_squared_error(y_val, y_pred_val_svr)
test_mse_svr = mean_squared_error(y_test, y_pred_test_svr)

train_rmse_svr = train_mse_svr ** 0.5
val_rmse_svr = val_mse_svr ** 0.5
test_rmse_svr = test_mse_svr ** 0.5

train_r2_svr = r2_score(y_train, y_pred_train_svr)
val_r2_svr = r2_score(y_val, y_pred_val_svr)
test_r2_svr = r2_score(y_test, y_pred_test_svr)

print(f"SVR - Training RMSE: {train_rmse_svr:.2f}")
print(f"SVR - Validation RMSE: {val_rmse_svr:.2f}")
print(f"SVR - Test RMSE: {test_rmse_svr:.2f}")

print(f"SVR - Training R^2: {train_r2_svr:.2f}")
print(f"SVR - Validation R^2: {val_r2_svr:.2f}")
print(f"SVR - Test R^2: {test_r2_svr:.2f}")

SVR - Training RMSE: 5891.37
SVR - Validation RMSE: 4932.95
SVR - Test RMSE: 8252.40
SVR - Training R^2: -0.02
SVR - Validation R^2: -0.04
SVR - Test R^2: -0.02


In [85]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the AdaBoost Regressor
ada_model = AdaBoostRegressor(random_state=42, n_estimators=200)
ada_model.fit(X_train, y_train)

# Make predictions
y_pred_train_ada = ada_model.predict(X_train)
y_pred_val_ada = ada_model.predict(X_val)
y_pred_test_ada = ada_model.predict(X_test)

# Evaluate the model
train_mse_ada = mean_squared_error(y_train, y_pred_train_ada)
val_mse_ada = mean_squared_error(y_val, y_pred_val_ada)
test_mse_ada = mean_squared_error(y_test, y_pred_test_ada)

train_rmse_ada = train_mse_ada ** 0.5
val_rmse_ada = val_mse_ada ** 0.5
test_rmse_ada = test_mse_ada ** 0.5

train_r2_ada = r2_score(y_train, y_pred_train_ada)
val_r2_ada = r2_score(y_val, y_pred_val_ada)
test_r2_ada = r2_score(y_test, y_pred_test_ada)

print(f"AdaBoost - Training RMSE: {train_rmse_ada:.2f}")
print(f"AdaBoost - Validation RMSE: {val_rmse_ada:.2f}")
print(f"AdaBoost - Test RMSE: {test_rmse_ada:.2f}")

print(f"AdaBoost - Training R^2: {train_r2_ada:.2f}")
print(f"AdaBoost - Validation R^2: {val_r2_ada:.2f}")
print(f"AdaBoost - Test R^2: {test_r2_ada:.2f}")

AdaBoost - Training RMSE: 9776.64
AdaBoost - Validation RMSE: 9973.53
AdaBoost - Test RMSE: 12209.27
AdaBoost - Training R^2: -1.82
AdaBoost - Validation R^2: -3.25
AdaBoost - Test R^2: -1.22


In [105]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


def train_xgb_for_file(file_path):
    data = pd.read_csv(file_path)
    data = data.drop(columns=['Unnamed: 0', 'Date'], errors='ignore')

    target_col = 'Price (₹/ton)' if 'Price (₹/ton)' in data.columns else 'Modal_x0020_Price'
    X_local = data.drop(columns=[target_col])
    y_local = data[target_col]

    # One-hot encode categorical features and align splits
    X_train_l, X_temp_l, y_train_l, y_temp_l = train_test_split(
        X_local, y_local, test_size=0.4, random_state=42
    )
    X_val_l, X_test_l, y_val_l, y_test_l = train_test_split(
        X_temp_l, y_temp_l, test_size=0.5, random_state=42
    )

    X_train_l = pd.get_dummies(X_train_l, drop_first=False)
    X_val_l = pd.get_dummies(X_val_l, drop_first=False).reindex(columns=X_train_l.columns, fill_value=0)
    X_test_l = pd.get_dummies(X_test_l, drop_first=False).reindex(columns=X_train_l.columns, fill_value=0)

    model_l = XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
    )
    model_l.fit(X_train_l, y_train_l)

    y_pred_train_l = model_l.predict(X_train_l)
    y_pred_val_l = model_l.predict(X_val_l)
    y_pred_test_l = model_l.predict(X_test_l)

    train_rmse_l = mean_squared_error(y_train_l, y_pred_train_l) ** 0.5
    val_rmse_l = mean_squared_error(y_val_l, y_pred_val_l) ** 0.5
    test_rmse_l = mean_squared_error(y_test_l, y_pred_test_l) ** 0.5

    train_r2_l = r2_score(y_train_l, y_pred_train_l)
    val_r2_l = r2_score(y_val_l, y_pred_val_l)
    test_r2_l = r2_score(y_test_l, y_pred_test_l)

    return {
        'Dataset': os.path.basename(file_path),
        'Rows': len(data),
        'Target': target_col,
        'Train_RMSE': round(float(train_rmse_l), 2),
        'Val_RMSE': round(float(val_rmse_l), 2),
        'Test_RMSE': round(float(test_rmse_l), 2),
        'Train_R2': round(float(train_r2_l), 4),
        'Val_R2': round(float(val_r2_l), 4),
        'Test_R2': round(float(test_r2_l), 4),
    }


requested_files = [
    'crop_dataset.csv',
    'crop_dataset_backup.csv',
]

if not os.path.exists('crop_dataset_backup.csv') and os.path.exists('crop_dataset_backup_before_augmentation.csv'):
    requested_files[1] = 'crop_dataset_backup_before_augmentation.csv'

available_files = [f for f in requested_files if os.path.exists(f)]
missing_files = [f for f in requested_files if not os.path.exists(f)]

results = [train_xgb_for_file(f) for f in available_files]

if results:
    print('Model training completed for:')
    for f in available_files:
        print(f'- {f}')
    print()
    print(pd.DataFrame(results))

if missing_files:
    print('\nMissing files:')
    for f in missing_files:
        print(f'- {f}')

Model training completed for:
- crop_dataset.csv
- crop_dataset_backup_before_augmentation.csv

                                       Dataset  Rows  ...  Val_R2  Test_R2
0                             crop_dataset.csv  7611  ...  0.9691   0.9710
1  crop_dataset_backup_before_augmentation.csv  7611  ...  0.9688   0.9663

[2 rows x 9 columns]
